# Constraints

$$ 
\min 3x+5y \\

2x+7y \le 28 \\
x \ge 4 \\
y \le 3 \\
x, y \ge 0 \\
$$

In [2]:
import pulp as p

Lp_prob = p.LpProblem('Problem', p.LpMinimize)
start_time = timeit.default_timer()

x = p.LpVariable("x", lowBound = 0)   # Create a variable x >= 0
y = p.LpVariable("y", lowBound = 0)   # Create a variable y >= 0

Lp_prob += 3*x + 5*y
Lp_prob += 2*x + 7*y <= 28
Lp_prob += x >= 4
Lp_prob += y <= 3

print(Lp_prob)

status = Lp_prob.solve()
solution_time = (timeit.default_timer() - start_time)

print(p.LpStatus[status])
print(p.value(x), p.value(y), p.value(Lp_prob.objective))  
print("Solution Time:", solution_time)


Problem:
MINIMIZE
3*x + 5*y + 0
SUBJECT TO
_C1: 2 x + 7 y <= 28

_C2: x >= 4

_C3: y <= 3

VARIABLES
x Continuous
y Continuous

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/charlie/Desktop/ml-all-phases/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/b0/mpb6l71d4lx_j9nt1h107vch0000gp/T/bd77c8961ac74707a0b4a67fea49e6cc-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/b0/mpb6l71d4lx_j9nt1h107vch0000gp/T/bd77c8961ac74707a0b4a67fea49e6cc-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 8 COLUMNS
At line 15 RHS
At line 19 BOUNDS
At line 20 ENDATA
Problem MODEL has 3 rows, 2 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 0 (-3) rows, 0 (-2) columns and 0 (-4) elements
Empty problem - 0 rows, 0 columns and 0 elements
Optimal - objective value 12
After Postsolve, objective 12

In [14]:
import pulp as p
import timeit

# Example Data

S = {
  1: {1, 2, 4},
  2: {3, 4, 5},
  3: {2, 5, 6},
  4: {1, 6, 7},
  5: {9, 10},
  6: {8, 10},
  7: {11},
  8: {8}
}
E = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11}
k = 3
W = [6, 7, 8, 6, 7, 8, 6, 7, 8, 6, 10]

def with_weights_ilp_basic(S, E, k, W):
  if len(W) != len(E):
    raise ValueError("W needs to be the same size as E")
  Lp_prob = p.LpProblem('MCLP With Weights', p.LpMaximize)
  start_time = timeit.default_timer()

  # defining x and y based on S and E, respectively
  x = p.LpVariable.dicts("x", S, cat = "Binary")
  y = p.LpVariable.dicts("y", E, cat = "Binary")
  w_map = dict(zip(sorted(E), W))

  # Define maximizing function
  Lp_prob += p.lpSum(w_map[j] * y[j] for j in E)

  # Limiting number of towers to k
  Lp_prob += p.lpSum(x[i] for i in S) <= k

  # A location can count only if a selected tower covers it
  for j in E:
    Lp_prob += p.lpSum(x[i] for i in S if j in S[i]) >= y[j]

  print(Lp_prob)

  status = Lp_prob.solve()
  solution_time = timeit.default_timer() - start_time

  print(p.LpStatus[status])

  if p.LpStatus[status] == "Optimal":
    print("Selected towers:", [i for i in S if p.value(x[i]) > 0.5])
    print("Covered locations:", [j for j in E if p.value(y[j]) > 0.5])
    print("Total covered:", p.value(Lp_prob.objective))

  print("Solution Time:", solution_time)



with_weights_ilp_basic(S, E, k, W)

MCLP_With_Weights:
MAXIMIZE
6*y_1 + 6*y_10 + 10*y_11 + 7*y_2 + 8*y_3 + 6*y_4 + 7*y_5 + 8*y_6 + 6*y_7 + 7*y_8 + 8*y_9 + 0.0
SUBJECT TO
_C1: x_1 + x_2 + x_3 + x_4 + x_5 + x_6 + x_7 + x_8 <= 3

_C2: x_1 + x_4 - y_1 >= 0

_C3: x_1 + x_3 - y_2 >= 0

_C4: x_2 - y_3 >= 0

_C5: x_1 + x_2 - y_4 >= 0

_C6: x_2 + x_3 - y_5 >= 0

_C7: x_3 + x_4 - y_6 >= 0

_C8: x_4 - y_7 >= 0

_C9: x_6 + x_8 - y_8 >= 0

_C10: x_5 - y_9 >= 0

_C11: x_5 + x_6 - y_10 >= 0

_C12: x_7 - y_11 >= 0

VARIABLES
0 <= x_1 <= 1 Integer
0 <= x_2 <= 1 Integer
0 <= x_3 <= 1 Integer
0 <= x_4 <= 1 Integer
0 <= x_5 <= 1 Integer
0 <= x_6 <= 1 Integer
0 <= x_7 <= 1 Integer
0 <= x_8 <= 1 Integer
0 <= y_1 <= 1 Integer
0 <= y_10 <= 1 Integer
0 <= y_11 <= 1 Integer
0 <= y_2 <= 1 Integer
0 <= y_3 <= 1 Integer
0 <= y_4 <= 1 Integer
0 <= y_5 <= 1 Integer
0 <= y_6 <= 1 Integer
0 <= y_7 <= 1 Integer
0 <= y_8 <= 1 Integer
0 <= y_9 <= 1 Integer

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/cha

/Users/charlie/Desktop/ml-all-phases/.venv/lib/python3.14/site-packages/pulp/pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


In [ ]:
import pulp as p
import timeit

# Example Data

S = {
  1: {1, 2, 4},
  2: {3, 4, 5},
  3: {2, 5, 6},
  4: {1, 6, 7},
  5: {9, 10},
  6: {8, 10},
  7: {11},
  8: {8}
}
E = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11}
k = 3

def no_weights_ilp_basic(S, E, k):
  Lp_prob = p.LpProblem('MCLP With Weights', p.LpMaximize)
  start_time = timeit.default_timer()

  # defining x and y based on S and E, respectively
  x = p.LpVariable.dicts("x", S, cat = "Binary")
  y = p.LpVariable.dicts("y", E, cat = "Binary")

  # Define maximizing function
  Lp_prob += p.lpSum(y[j] for j in E)

  # Limiting number of towers to k
  Lp_prob += p.lpSum(x[i] for i in S) <= k

  # A location can count only if a selected tower covers it
  for j in E:
    Lp_prob += p.lpSum(x[i] for i in S if j in S[i]) >= y[j]

  print(Lp_prob)

  status = Lp_prob.solve()
  solution_time = timeit.default_timer() - start_time

  print(p.LpStatus[status])

  if p.LpStatus[status] == "Optimal":
    print("Selected towers:", [i for i in S if p.value(x[i]) > 0.5])
    print("Covered locations:", [j for j in E if p.value(y[j]) > 0.5])
    print("Total covered:", p.value(Lp_prob.objective))

  print("Solution Time:", solution_time)



with_weights_ilp_basic(S, E, k)